In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# read the file from bronze layer
payments_df = spark.read.table("retail_lakehouse.bronze.payments")

display(payments_df)
payments_df.printSchema()


In [0]:
# standardize the column names
payments_clean_df = (
    payments_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "payment_id",
        F.trim(F.col("payment_id"))
    )
    .withColumn(
        "payment_amount",
        F.col("payment_amount").cast("double")
    )
    .withColumn(
        "payment_method",
        F.upper(F.col("payment_method"))
    )
    .withColumn(
        "payment_status",
        F.upper(F.col("payment_status"))
    )
    .withColumn(
        "payment_timestamp",
        F.to_timestamp(F.col("payment_timestamp"), "yyyy-MM-dd HH:mm:ss")
    )
)

display(payments_clean_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

payment_window = Window.partitionBy("payment_id")

payments_validation_df = (
    payments_clean_df

    .withColumn(
        "dq_payment_id_not_null",
        F.col("payment_id").isNotNull()
    )

    .withColumn(
        "dq_payment_id_count",
        F.count("*").over(payment_window)
    )

    .withColumn(
        "dq_payment_id_unique",
        F.col("dq_payment_id_count") == 1
    )

    .withColumn(
        "dq_order_id_not_null",
        F.col("order_id").isNotNull()
    )

    .withColumn(
        "dq_payment_amount",
        F.col("payment_amount") > 0
    )

    .withColumn(
        "dq_payment_method",
        F.col("payment_method").isNotNull()
    )

    .withColumn(
        "dq_payment_status",
        F.col("payment_status").isNotNull()
    )

    .withColumn(
        "dq_payment_timestamp",
        F.col("payment_timestamp").isNotNull()
    )
)

In [0]:
orders_lookup_df = (
    spark.table("retail_lakehouse.sliver.orders")
    .select(
        F.col("order_id").alias("ref_order_id")
    )
    .dropDuplicates()
)

payments_order_checkup_df = (
    payments_validation_df
    .join(
        orders_lookup_df,
        payments_validation_df["order_id"] ==
        orders_lookup_df["ref_order_id"],
        "left"
    )
    .withColumn(
        "dq_order_exists",
        F.col("ref_order_id").isNotNull()
    )
)

In [0]:
print("Before join:", payments_validation_df.count())
print("After join:", payments_order_checkup_df.count())

In [0]:
# orders total

orders_total_df = (
    spark.table("retail_lakehouse.sliver.order_items")
    .groupBy("order_id")
    .agg(
        F.sum("line_amount").alias("expected_payment_amount")
    )
)

display(orders_total_df)

In [0]:
# compare the order_amount

payments_amount_check_df = (
    payments_order_checkup_df
    .join(
        orders_total_df,
        on="order_id",
        how="left"
    )
    .withColumn(
        "dq_payments_amount_match",
        F.abs(
            F.col("payment_amount") - 
            F.col("expected_payment_amount")
        ) < 0.01
    )
)

display(payments_amount_check_df)

In [0]:
payment_final_df = (
    payments_amount_check_df
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_payment_id_not_null") &
            F.col("dq_payment_id_unique") &
            F.col("dq_order_id_not_null") &
            F.col("dq_payment_amount") &
            F.col("dq_payment_method") &
            F.col("dq_payment_status") &
            F.col("dq_payment_timestamp") &
            F.col("dq_order_exists") &
            F.col("dq_payments_amount_match"),
            "PASS"
        ).otherwise("FAIL")
    )
)

In [0]:
valid_df = (
    payment_final_df
    .filter(F.col("dq_status") == "PASS")
)

invalid_df = (
    payment_final_df
    .filter(F.col("dq_status") == "FAIL")
)

In [0]:
silver_payments_df = (
    valid_df
    .select(
        "order_id",
        "payment_amount",
        "payment_id",
        "payment_method",
        "payment_status",
        "payment_timestamp"
    )
)

In [0]:
(
    valid_df
    .select(
        "order_id",
        "payment_amount",
        "payment_id",
        "payment_method",
        "payment_status",
        "payment_timestamp"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_lakehouse.sliver.payments")
)

In [0]:
(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_lakehouse.sliver.payments_quarantine")
)

In [0]:
%sql

SELECT * FROM payments_quarantine